# Resultados de las tres etapas

Una fila por registro de man_val: manual, benchmark y BO.
Los datos faltantes quedan vacíos. Solo muestra la tabla.

# Librerías

Se utilizan pandas, NumPy, PyYAML y rdata, como en las notebooks Python del repositorio.
No requiere LightGBM, Optuna, R ni GPU.

In [1]:
import os
import json
import sqlite3
import numpy as np
import pandas as pd
import yaml
import rdata
from IPython.display import display

# URLS

In [2]:
BASE_URL = os.path.join(
    os.path.expanduser("~"),
    "code", "DMEyF", "dmeyf2026"
)
# En la VM de Google Cloud, reemplazar BASE_URL por:
# BASE_URL = os.path.join(os.path.expanduser("~"), "buckets", "b1", "dmeyf2026")

EXP_URL = os.path.join(BASE_URL, "DATA", "EXP")
carpeta_man_val = os.path.join(BASE_URL, "ensembles", "man_val")
archivo_validaciones = os.path.join(carpeta_man_val, "validaciones_manuales.csv")

# Funciones

Se toma el mejor modelo del benchmark y se busca su trial en la BO original.
El dataset de la BO se lee de su documentación, sin deducirlo de etapas posteriores.

In [3]:
def convertir_r(valor):
    if isinstance(valor, dict):
        return {str(k): convertir_r(v) for k, v in valor.items()}
    if isinstance(valor, pd.DataFrame):
        return convertir_r(valor.to_dict(orient="list"))
    if isinstance(valor, np.ndarray):
        if valor.size == 1:
            return convertir_r(valor.item())
        return [convertir_r(v) for v in valor.tolist()]
    if isinstance(valor, (list, tuple)):
        return [convertir_r(v) for v in valor]
    if isinstance(valor, np.generic):
        return convertir_r(valor.item())
    if isinstance(valor, float) and np.isfinite(valor) and valor.is_integer():
        return int(valor)
    return valor


def leer_rds(archivo):
    if not os.path.exists(archivo):
        return {}
    return convertir_r(rdata.read_rds(archivo))


def leer_parametros(carpeta):
    archivo = os.path.join(carpeta, "PARAM.yml")
    if os.path.exists(archivo):
        with open(archivo, encoding="utf-8") as f:
            return convertir_r(yaml.safe_load(f) or {})
    return leer_rds(os.path.join(carpeta, "PARAM.rds"))


def identificador(valor):
    if valor is None or pd.isna(valor):
        return None
    return str(convertir_r(valor))


def semillas_manual(registro):
    seeds = registro.get("seeds")
    if pd.notna(seeds) and str(seeds).strip():
        return len(set(s.strip() for s in str(seeds).split(",") if s.strip()))
    parametros = leer_rds(os.path.join(
        carpeta_man_val, f"{int(registro['id_validacion'])}.rds"
    ))
    if "num_iterations" in parametros:
        return 1
    if parametros and all(isinstance(v, dict) for v in parametros.values()):
        return len(parametros)
    return None


def mostrar_hp(parametros):
    if parametros is None:
        return None
    return "; ".join(f"{k}={convertir_r(v)}" for k, v in parametros.items())


def nombre_metrica(valor):
    if isinstance(valor, list) and len(valor) == 1:
        valor = valor[0]
    if not isinstance(valor, str):
        return None
    return {"average_precision": "ap", "ap": "ap", "auc": "auc", "ganancia": "ganancia"}.get(valor.lower())

In [4]:
def leer_bo(experimento, trial=None, resultado=None, seleccionar_mejor=False):
    carpeta = os.path.join(EXP_URL, experimento)
    param = leer_parametros(carpeta)
    punto = {}
    archivo_log = os.path.join(carpeta, "BO_log.txt")

    if os.path.exists(archivo_log):
        log = pd.read_csv(archivo_log, sep="\t")
        log["fila_origen"] = np.arange(1, len(log) + 1)
        if "state" in log:
            log = log.loc[log["state"].eq("COMPLETE")]
        campo_metrica = next((k for k in ("metrica", "value", "y") if k in log), None)
        campo_id = next(k for k in ("iter", "number", "fila_origen") if k in log)
        if trial is not None:
            seleccion = log.loc[log[campo_id].map(identificador).eq(identificador(trial))]
        elif seleccionar_mejor and campo_metrica is not None:
            scores = pd.to_numeric(log[campo_metrica], errors="coerce")
            seleccion = log.loc[[scores.idxmax()]] if scores.notna().any() else log.iloc[:0]
        else:
            seleccion = log.iloc[:0]
        if not seleccion.empty:
            punto = convertir_r(seleccion.iloc[0].to_dict())
            trial = punto[campo_id]
            resultado = punto.get(campo_metrica, resultado)

    # La DB se abre solo para lectura. Sus distribuciones permiten distinguir
    # rangos constantes de HP realmente optimizados, aunque ambos estén en el log.
    distribuciones = {}
    configuracion = {}
    archivo_db = os.path.join(carpeta, "optuna_study.db")
    if os.path.exists(archivo_db):
        conexion = sqlite3.connect(f"file:{os.path.abspath(archivo_db)}?mode=ro", uri=True)
        try:
            estudios = conexion.execute("SELECT study_id, study_name FROM studies").fetchall()
            estudio = next((s for s in estudios if s[1] == experimento), None)
            if estudio is None and len(estudios) == 1:
                estudio = estudios[0]
            if estudio is not None:
                attrs = dict(conexion.execute(
                    "SELECT key, value_json FROM study_user_attributes WHERE study_id=?",
                    (estudio[0],)
                ).fetchall())
                configuracion = json.loads(attrs.get("configuracion_bo", "{}"))
                if "number" in punto:
                    filas = conexion.execute(
                        "SELECT p.param_name, p.distribution_json FROM trial_params p "
                        "JOIN trials t ON p.trial_id=t.trial_id "
                        "WHERE t.study_id=? AND t.number=?",
                        (estudio[0], int(punto["number"]))
                    ).fetchall()
                    distribuciones = {k: json.loads(v)["attributes"] for k, v in filas}
        finally:
            conexion.close()

    param_fijos = configuracion.get("param_fijos", param.get("lgbm", {}).get("param_fijos", {}))
    rangos_bo = configuracion.get("rangos_bo", param.get("hyperparametertuning", {}).get("rangos_bo", {}))
    hp_trial = {k[7:]: v for k, v in punto.items() if k.startswith("params_") and pd.notna(v)}

    if not hp_trial and "y" in punto:
        # Los logs de mlrMBO de z494 guardan los HP variables junto con y.
        mejores_hp = param.get("out", {}).get("lgbm", {}).get("mejores_hiperparametros", {})
        nombres = set(param_fijos) | set(mejores_hp)
        hp_trial = {k: v for k, v in punto.items() if k in nombres and pd.notna(v)}
    if not hp_trial and rangos_bo:
        hp_trial = {k: punto[k] for k in rangos_bo if k in punto and pd.notna(punto[k])}

    optimizados = {}
    for nombre, valor in hp_trial.items():
        limites = rangos_bo.get(nombre)
        dist = distribuciones.get(nombre, {})
        constante = (
            (limites is not None and len(limites) == 2 and limites[0] == limites[1])
            or ("low" in dist and dist["low"] == dist["high"])
            or ("choices" in dist and len(dist["choices"]) == 1)
        )
        if not constante:
            optimizados[nombre] = valor

    optimizados = mostrar_hp(optimizados) if hp_trial else None

    dataset = configuracion.get("dataset", param.get("dataset"))
    if isinstance(dataset, str):
        dataset = os.path.basename(dataset)
    semilla = configuracion.get("semilla", param.get("semilla_primigenia", param_fijos.get("seed")))
    if seleccionar_mejor and resultado is None and trial is None:
        resultado = param.get("out", {}).get("lgbm", {}).get("y")

    return {
        "bo_resultado": resultado,
        "bo_semillas": 1 if semilla is not None else None,
        "bo_dataset": dataset,
        "bo_metrica": nombre_metrica(param_fijos.get("metric")),
        "bo_hp": optimizados,
        "experimento": experimento,
    }

# Lectura y reconstrucción

In [5]:
validaciones = pd.read_csv(archivo_validaciones)
filas = []

for _, registro in validaciones.iterrows():
    fila = {
        "man_val_resultado": registro.get("ganancia"),
        "man_val_semillas": semillas_manual(registro),
        "man_val_dataset": registro.get("dataset"),
        "man_val_metrica": "ganancia" if pd.notna(registro.get("ganancia")) else None,
        "benchmark_resultado": None,
        "benchmark_semillas": None,
        "benchmark_dataset": None,
        "benchmark_metrica": None,
        "bo_resultado": None,
        "bo_semillas": None,
        "bo_dataset": None,
        "bo_metrica": None,
        "bo_hp": None,
        "id_rds": int(registro["id_validacion"]),
        "benchmark": None,
        "experimento": None,
    }

    referencia = registro.get("benchmark_original")
    if pd.notna(referencia) and str(referencia).strip():
        referencia = str(referencia)
        if os.path.basename(referencia).startswith("benchmark"):
            fila["benchmark"] = os.path.basename(referencia).removeprefix("benchmark_multisemilla_lags")
            carpeta = os.path.join(EXP_URL, referencia)
            param_bench = leer_parametros(carpeta)
            semillas = param_bench.get("semillas")
            if semillas is not None:
                fila["benchmark_semillas"] = len(semillas) if isinstance(semillas, list) else 1
            else:
                fila["benchmark_semillas"] = param_bench.get("m_semillas")
                archivo_semillas = os.path.join(carpeta, "semillas_usadas.tsv")
                if fila["benchmark_semillas"] is None and os.path.exists(archivo_semillas):
                    tb_semillas = pd.read_csv(archivo_semillas, sep="\t")
                    fila["benchmark_semillas"] = int(tb_semillas["uso"].str.startswith("entrenamiento_").sum())
            fila["benchmark_dataset"] = param_bench.get("dataset")

            archivo_mejores = os.path.join(carpeta, "mejor_corte_por_modelo.tsv")
            if os.path.exists(archivo_mejores):
                mejores = pd.read_csv(archivo_mejores, sep="\t")
                if mejores["ganancia_total_promedio"].notna().any():
                    mejor = mejores.loc[mejores["ganancia_total_promedio"].idxmax()]
                    fila["benchmark_resultado"] = mejor["ganancia_total_promedio"]
                    fila["benchmark_dataset"] = mejor.get("dataset_bench", fila["benchmark_dataset"])
                    fila["benchmark_metrica"] = "ganancia"
                    experimento = mejor.get("experimento_origen")
                    if pd.notna(experimento):
                        fila.update(leer_bo(
                            str(experimento),
                            mejor.get("id_origen"),
                            mejor.get("metrica_origen")
                        ))
        else:
            # rds_sin_validacion guarda el nombre de la BO en benchmark_original.
            fila.update(leer_bo(referencia, seleccionar_mejor=True))

    filas.append(fila)

resultados_3_etapas = pd.DataFrame(filas)

# Tabla

HP muestra los explorados y sus valores. RDS es el ID del archivo <id>.rds.
Las referencias quedan al final y el benchmark muestra solo fecha y hora.

In [6]:
tabla = resultados_3_etapas.copy()
tabla.columns = pd.MultiIndex.from_tuples([
    ("Manual", "Resultado"), ("Manual", "Semillas"),
    ("Manual", "Dataset"), ("Manual", "Métrica"),
    ("Benchmark", "Resultado"), ("Benchmark", "Semillas"),
    ("Benchmark", "Dataset"), ("Benchmark", "Métrica"),
    ("BO", "Resultado"), ("BO", "Semillas"),
    ("BO", "Dataset"), ("BO", "Métrica"), ("BO", "HP"),
    ("", "RDS"), ("", "Benchmark"), ("", "Experimento"),
])

formatos = {
    ("Manual", "Resultado"): "{:,.0f}",
    ("Benchmark", "Resultado"): "{:,.0f}",
    ("BO", "Resultado"): "{:.6f}",
    ("Manual", "Semillas"): "{:.0f}",
    ("Benchmark", "Semillas"): "{:.0f}",
    ("BO", "Semillas"): "{:.0f}",
    ("", "RDS"): "{:.0f}",
}

display(
    tabla.style
    .format(formatos, na_rep="")
    .hide(axis="index")
    .set_properties(**{"text-align": "left", "white-space": "normal"})
    .set_properties(
        subset=[("BO", "HP")],
        **{"max-width": "28em", "overflow-wrap": "anywhere"}
    )
)

# TAbla compacta

In [9]:
abreviaturas = {
    "learning_rate": "LR",
    "feature_fraction": "FF",
    "min_sum_hessian_in_leaf": "hessian",
    "num_iterations": "n_iter",
    "num_leaves": "n_leav",
    "min_data_in_leaf": "min_leaf",
    "scale_pos_weight": "SPW",
    "feature_fraction_bynode": "FF_node",
    "lambda_l1": "L1",
    "lambda_l2": "L2",
    "min_gain_to_split": "gain",
    "bagging_fraction": "BF",
    "bagging_freq": "Bfreq",
    "pos_bagging_fraction": "BF_pos",
    "neg_bagging_fraction": "BF_neg",
    "max_depth": "depth",
    "max_bin": "bin",
    "extra_trees": "ET",
    "prob_corte": "p_corte",
}

def nombres_hp(valor):
    if pd.isna(valor):
        return None
    return {parte.split("=", 1)[0].strip() for parte in valor.split(";") if parte.strip()}

hp_por_fila = resultados_3_etapas["bo_hp"].map(nombres_hp)
todos_hp = set().union(*(hp for hp in hp_por_fila if hp is not None))
columnas_hp = [hp for hp in abreviaturas if hp in todos_hp]
columnas_hp += sorted(todos_hp - set(abreviaturas))

tabla_compacta = resultados_3_etapas[[
    "man_val_resultado", "man_val_dataset", "benchmark_resultado",
    "bo_metrica", "experimento",
]].copy()
tabla_compacta.columns = [
    "Gan. manual", "Dataset", "Gan. bench", "Métrica BO", "Experimento"
]

for hp in columnas_hp:
    tabla_compacta[abreviaturas.get(hp, hp)] = hp_por_fila.map(
        lambda explorados: pd.NA if explorados is None else int(hp in explorados)
    ).astype("Int64")

display(
    tabla_compacta.style
    .format({"Gan. manual": "{:,.0f}", "Gan. bench": "{:,.0f}"}, na_rep="")
    .hide(axis="index")
    .set_table_attributes('style="font-size:12px;"')
    .set_properties(**{"text-align": "center"})
    .set_properties(
        subset=["Dataset"],
        **{
            "width": "14em", "max-width": "14em",
            "white-space": "normal", "overflow-wrap": "anywhere",
            "text-align": "left",
        }
    )
)

Gan. manual,Dataset,Gan. bench,Métrica BO,Experimento,LR,FF,hessian,n_iter,n_leav,min_leaf,SPW,p_corte
"471,405,000",competencia_01_ternaria_lags.csv,"464,200,000",ap,LAG-002,1,1,1,1,1,0,0,0
"452,017,500",competencia_01_ternaria_lf_k6_perdida4.csv,"464,200,000",ap,LAG-002,1,1,1,1,1,0,0,0
"397,952,500",competencia_01_ternaria_lags.csv,"464,200,000",ap,LAG-002,1,1,1,1,1,0,0,0
"471,872,500",competencia_01_ternaria_lags.csv,"455,510,000",ap,LAG-001,1,1,1,1,1,0,0,0
"457,242,500",competencia_01_ternaria_lf_k6_perdida4.csv,"455,510,000",ap,LAG-001,1,1,1,1,1,0,0,0
"460,295,000",competencia_01_ternaria_lf_k6_perdida4.csv,"451,000,000",auc,LAG-003,1,1,1,1,1,0,0,0
"466,400,000",competencia_01_ternaria_lags.csv,"451,000,000",auc,LAG-003,1,1,1,1,1,0,0,0
"464,007,500",competencia_01_ternaria_lags.csv,"463,760,000",auc,LAG-003,1,1,1,1,1,0,0,0
"460,267,500",competencia_01_ternaria_lf_k6_perdida4.csv,"463,760,000",auc,LAG-003,1,1,1,1,1,0,0,0
"452,017,500",competencia_01_ternaria_lf_k6_perdida4.csv,"464,200,000",ap,LAG-002,1,1,1,1,1,0,0,0


# Rangos parametros USADOS

In [8]:
filas = []
carpeta_exp = os.path.join(BASE_URL, "DATA", "EXP")

for experimento in sorted(os.listdir(carpeta_exp)):
    archivo = os.path.join(carpeta_exp, experimento, "BO_log.txt")
    if not os.path.isfile(archivo):
        continue

    log = pd.read_csv(archivo, sep="\t")
    for columna in ("min_data_in_leaf", "params_min_data_in_leaf"):
        if columna in log.columns:
            valores = pd.to_numeric(log[columna], errors="coerce")
            filas.append({
                "Experimento": experimento,
                "Min": valores.min(),
                "Max": valores.max(),
            })
            break

display(pd.DataFrame(filas))

,Experimento,Min,Max
0,HT4080,111,7998
1,HT4940,1,7738
2,HT4941,50,1981
3,HT4942,0,1996
4,LAG-008,0,500
5,LAG-009,100,1000
6,LAG-010,0,1000
7,LAG-011,0,0
8,LAG-012,0,0
9,LAG-013,0,0
